# 07 - Train Multitask (kepala bantu perilaku + sudut)

In [ ]:
import os, sys, zipfile
from pathlib import Path

# Cari root repo: di laptop = folder di atas notebooks/, di Kaggle = dataset mlc-src
if os.path.exists("/kaggle/input"):
    hits = list(Path("/kaggle/input").rglob("src/data.py"))
    if hits:
        ROOT = hits[0].parent.parent
    else:  # dataset masih berupa zip
        ROOT = Path("/kaggle/working/_repo")
        for z in Path("/kaggle/input").rglob("*.zip"):
            if z.stem in ("src", "configs", "annotations"):
                zipfile.ZipFile(z).extractall(ROOT / z.stem)
else:
    ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))

from src.data import ON_KAGGLE, get_path, load_config
from src.seed import seed_everything

cfg = load_config()
seed_everything(cfg["SEED"])
print("Kaggle:", ON_KAGGLE, "| data:", get_path(cfg, "raw"))

## Setelan, index (loncat 5), embedding

In [ ]:

import json
import time

import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader

from src.data import get_path
from src.index import bangun_index
from src.metrics import cari_threshold, ringkas_teks, skor, skor_per_fold
from src.models import (augmentasi, jumlah_parameter, muat_semua_embedding,
                        muat_test_mentah, peta_perilaku, prediksi_batch,
                        saring_jendela, susun_jendela)
from src.multitask import DataJendelaMultitask, ModelMultitask

METRIK = cfg["METRIC"]          # f1_macro; jangan pakai default "f1" di src/metrics.py
SEQ_LEN = cfg["SEQ_LEN"]
SEED = cfg["SEED"]
LONCAT = 5                      # index lebih padat daripada notebook 04 (loncat 10)
SEEDS = [SEED, SEED + 1, SEED + 2]
EPOCH_MAKS = 18                 # tahap A: panjang kurva validasi
SMOKE = False                   # True = uji cepat (2 fold, 1 seed, 2 epoch, sedikit jendela),
                                # bukan hasil nyata. Hasilnya dipisah lewat akhiran TAG.
TAG = "_smoke" if SMOKE else ""
BATCH = 256
LR, WD = 3e-4, 0.05
BOBOT_PERILAKU = 0.3            # bobot loss kepala bantu perilaku (FAQ 34)
BOBOT_SUDUT = 0.1               # bobot loss kepala bantu sudut kamera
AUG = dict(noise=0.05, drop_fitur=0.1, mask_frame=0.2)   # hanya saat training
AMP = torch.cuda.is_available()
PEKERJA = 2 if ON_KAGGLE else 0
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device, "| metrik:", METRIK, "| mixed precision:", AMP)

dir_fitur = get_path(cfg, "features", read=True)
DIR_OUT = get_path(cfg, "oof").parent
DIR_MODEL = get_path(cfg, "models")
for d in (DIR_OUT / "kurva", DIR_OUT / "parsial", DIR_MODEL):
    d.mkdir(parents=True, exist_ok=True)

# Index disimpan supaya notebook bisa dilanjutkan tanpa membaca ulang semua video.
p_idx = DIR_OUT / f"index_potongan_loncat{LONCAT}.csv"
if p_idx.exists():
    idx = pd.read_csv(p_idx)
    print("index dibaca dari:", p_idx)
else:
    idx = bangun_index(cfg, loncat=LONCAT)
    idx.to_csv(p_idx, index=False)
    print("index dibangun ->", p_idx)

idx = idx[idx["dipakai"]].reset_index(drop=True)     # buang potongan tepat setengah (label tak jelas)
emb, hilang = muat_semua_embedding(dir_fitur, idx["video"].unique())
idx, lap = saring_jendela(idx, emb, seq_len=SEQ_LEN)
print("jendela dipakai:", len(idx),
      "| dibuang karena melewati panjang embedding:", lap["dibuang_lewat"],
      "| dibuang karena embedding tidak ada:", lap["dibuang_tanpa_embedding"])
print("embedding di memori:",
      f"{sum(a.nbytes for a in emb.values()) / 1e9:.2f} GB", "untuk", len(emb), "video")
if hilang:
    print("video tanpa embedding (maks 5):", hilang[:5])

PETA = peta_perilaku(idx)                            # kelas kepala bantu perilaku
PETA_SUDUT = peta_perilaku(idx, kolom="sudut")       # kelas kepala bantu sudut kamera
daftar_fold = sorted(idx["fold"].unique().tolist())
val_mask = idx["grid30"].to_numpy(bool)              # validasi = potongan tanpa geser (bentuk test)
print("fold:", daftar_fold, "| rasio label 1:", round(float(idx["label"].mean()), 4),
      "| jendela validasi (grid30):", int(val_mask.sum()),
      "| kelas perilaku:", len(PETA), "| kelas sudut:", len(PETA_SUDUT))

if SMOKE:        # uji cepat: 2 fold, 1 seed, sedikit jendela. Semua jendela validasi
    N_SMOKE = 512                 # fold terpilih tetap dipakai, sisanya diambil berimbang.
    daftar_fold = daftar_fold[:2]
    SEEDS, EPOCH_MAKS = [SEED], 2
    pilih = idx[idx["fold"].isin(daftar_fold)]
    inti = pilih[pilih["grid30"]]
    sisa = pilih[~pilih["grid30"]]
    per_grup = max(1, (N_SMOKE - len(inti)) // max(1, 2 * len(daftar_fold)))
    ambil = [inti] + [g.sample(min(len(g), per_grup), random_state=SEED)
                      for _, g in sisa.groupby(["fold", "label"])]
    idx = (pd.concat(ambil).sort_values(["video", "mulai_frame"]).reset_index(drop=True))
    val_mask = idx["grid30"].to_numpy(bool)
    PETA = peta_perilaku(idx)
    PETA_SUDUT = peta_perilaku(idx, kolom="sudut")
    print("SMOKE: fold", daftar_fold, "| seeds", SEEDS, "| epoch maks", EPOCH_MAKS,
          "| jendela", len(idx), "| validasi", int(val_mask.sum()),
          "| rasio label 1:", round(float(idx["label"].mean()), 4),
          "| kelas perilaku:", len(PETA), "| kelas sudut:", len(PETA_SUDUT))

Xte, ids_te = muat_test_mentah(dir_fitur, seq_len=SEQ_LEN)
DIM = int(Xte.shape[2])
print("test:", Xte.shape, "| dim fitur:", DIM)

## Model: alur dan jumlah parameter

In [ ]:

cek = ModelMultitask(dim_fitur=DIM, seq_len=SEQ_LEN, n_perilaku=len(PETA),
                     n_sudut=len(PETA_SUDUT))
N_PARAM = jumlah_parameter(cek)
print(cek)
print("jumlah parameter:", f"{N_PARAM:,}")
assert N_PARAM < 2_000_000, f"{N_PARAM} parameter, batasnya 2 juta"
del cek

## Satu kali latih (dipakai tahap A dan B)

In [ ]:

def buat_model():
    return ModelMultitask(dim_fitur=DIM, seq_len=SEQ_LEN, n_perilaku=len(PETA),
                          n_sudut=len(PETA_SUDUT)).to(device)


def pemuat(idx_tr):
    data = DataJendelaMultitask(idx_tr, emb, seq_len=SEQ_LEN, peta=PETA,
                                peta_sudut=PETA_SUDUT)
    return DataLoader(data, batch_size=BATCH, shuffle=True, drop_last=False,
                      num_workers=PEKERJA, pin_memory=(device.type == "cuda"),
                      persistent_workers=PEKERJA > 0)


def latih(idx_tr, n_epoch, seed, Xva=None, yva=None, cetak=False):
    """Latih satu model. Kalau Xva diberikan, skor validasi dihitung tiap epoch (tahap A).
    Return (model, kurva skor per epoch, detik per epoch)."""
    seed_everything(seed)
    model = buat_model()
    loader = pemuat(idx_tr)
    n_pos = int((idx_tr["label"] == 1).sum())
    bobot_pos = torch.tensor([(len(idx_tr) - n_pos) / max(n_pos, 1)], device=device)  # dari train fold saja
    rugi_biner = nn.BCEWithLogitsLoss(pos_weight=bobot_pos)
    rugi_perilaku = nn.CrossEntropyLoss()
    rugi_sudut = nn.CrossEntropyLoss()
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WD)
    jadwal = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=max(1, n_epoch * len(loader)))
    try:
        skaler = torch.amp.GradScaler("cuda", enabled=AMP)
    except TypeError:                                  # torch lama
        skaler = torch.cuda.amp.GradScaler(enabled=AMP)

    kurva, t0 = [], time.time()
    for ep in range(n_epoch):
        model.train()
        for x, y, k, sd in loader:
            x = x.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True).float()
            k = k.to(device, non_blocking=True)
            sd = sd.to(device, non_blocking=True)
            x = augmentasi(x, **AUG)                   # augmentasi HANYA di batch training
            opt.zero_grad(set_to_none=True)
            with torch.autocast("cuda", enabled=AMP):
                logit, aux = model(x)
                # total = BCE(cheat) + 0.3*CE(perilaku) + 0.1*CE(sudut)
                rugi = rugi_biner(logit.float(), y)
                if aux["perilaku"] is not None:
                    rugi = rugi + BOBOT_PERILAKU * rugi_perilaku(aux["perilaku"].float(), k)
                if aux["sudut"] is not None:
                    rugi = rugi + BOBOT_SUDUT * rugi_sudut(aux["sudut"].float(), sd)
            skaler.scale(rugi).backward()
            skaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            skaler.step(opt)
            skaler.update()
            jadwal.step()
        if Xva is not None:
            prob = prediksi_batch(model, Xva, device, amp=AMP)
            t, s = cari_threshold(yva, prob, metrik=METRIK)
            kurva.append(s)
            if cetak:
                print(f"    epoch {ep + 1:2d}/{n_epoch} {METRIK}={s:.4f} (thr={t:.2f})")
    return model, kurva, (time.time() - t0) / max(1, n_epoch)

## Tahap A: kurva validasi untuk memilih jumlah epoch

Satu seed per fold dilatih `EPOCH_MAKS` epoch. Jumlah epoch dipilih **sekali** dari rata-rata
kurva semua fold, lalu dipakai tetap di tahap B. Jadi tidak ada "epoch terbaik per fold" yang
membuat OOF terlihat lebih bagus daripada seharusnya. Model tahap A dibuang; yang dipakai hanya
angka epochnya.

In [ ]:

kurva_fold, detik_epoch = {}, []
for f in daftar_fold:
    p_kurva = DIR_OUT / "kurva" / f"kurva_mt_f{f}{TAG}.csv"
    if p_kurva.exists():                               # lanjutan sesi yang terputus
        kurva_fold[f] = pd.read_csv(p_kurva)["skor"].to_numpy()
        print(f"fold {f}: kurva sudah ada, dilewati")
        continue
    tr = idx[idx["fold"] != f]
    va = idx[(idx["fold"] == f) & val_mask]
    Xva = susun_jendela(va, emb, seq_len=SEQ_LEN)
    yva = va["label"].to_numpy(int)
    print(f"fold {f}: train {len(tr)} jendela | validasi {len(va)} (rasio 1 = {yva.mean():.3f})")
    _, kurva, dt = latih(tr, EPOCH_MAKS, SEED, Xva=Xva, yva=yva, cetak=True)
    kurva_fold[f] = np.asarray(kurva)
    detik_epoch.append(dt)
    pd.DataFrame({"epoch": np.arange(1, len(kurva) + 1), "skor": kurva}).to_csv(p_kurva, index=False)
    del Xva

panjang = min(len(v) for v in kurva_fold.values())
rata = np.mean([kurva_fold[f][:panjang] for f in daftar_fold], axis=0)
puncak = float(rata.max())
EPOCH_TETAP = int(np.argmax(rata >= puncak - 0.002)) + 1   # epoch terkecil yang sudah setara puncak
print("kurva rata-rata semua fold:", np.round(rata, 4).tolist())
if SMOKE:
    EPOCH_TETAP = 2
print(f"EPOCH_TETAP = {EPOCH_TETAP} ({METRIK} rata-rata {rata[EPOCH_TETAP - 1]:.4f}, puncak {puncak:.4f})")

pd.concat([pd.DataFrame({"fold": f, "epoch": np.arange(1, len(v) + 1), "skor": v})
           for f, v in kurva_fold.items()], ignore_index=True).to_csv(
    DIR_OUT / f"kurva_validasi_07{TAG}.csv", index=False)
print("kurva tersimpan:", DIR_OUT / f"kurva_validasi_07{TAG}.csv",
      "| detik/epoch:", [round(d, 1) for d in detik_epoch] or "(semua dari cache)")

## Tahap B: 5 fold x 3 seed dengan epoch tetap

Bobot dan prediksi disimpan per model, jadi kalau sesi Kaggle terputus notebook bisa dijalankan
ulang dan model yang bobotnya sudah ada akan dilewati.

In [ ]:

baris_oof, prob_test, waktu_model = [], {}, []
for f in daftar_fold:
    va = idx[(idx["fold"] == f) & val_mask]
    tr = idx[idx["fold"] != f]
    Xva = None
    for s in SEEDS:
        p_bobot = DIR_MODEL / f"multitask_f{f}_s{s}{TAG}.pt"
        p_oof = DIR_OUT / "parsial" / f"oof_mt_f{f}_s{s}{TAG}.csv"
        p_te = DIR_OUT / "parsial" / f"test_mt_f{f}_s{s}{TAG}.npy"
        if p_bobot.exists() and p_oof.exists() and p_te.exists():
            baris_oof.append(pd.read_csv(p_oof))
            prob_test[(f, s)] = np.load(p_te)
            print(f"fold {f} seed {s}: sudah ada, dilewati")
            continue
        if Xva is None:
            Xva = susun_jendela(va, emb, seq_len=SEQ_LEN)
        t0 = time.time()
        model, _, dt = latih(tr, EPOCH_TETAP, s)
        torch.save({"state_dict": model.state_dict(), "fold": f, "seed": s, "epoch": EPOCH_TETAP,
                    "dim_fitur": DIM, "seq_len": SEQ_LEN, "peta_perilaku": PETA,
                    "peta_sudut": PETA_SUDUT}, p_bobot)
        d = va[["video", "mulai_frame", "fold", "label"]].copy()
        d["prob"] = prediksi_batch(model, Xva, device, amp=AMP)    # eval + no_grad di dalam
        d["seed"] = s
        d.to_csv(p_oof, index=False)
        baris_oof.append(d)
        prob_test[(f, s)] = prediksi_batch(model, Xte, device, amp=AMP)
        np.save(p_te, prob_test[(f, s)])
        waktu_model.append(time.time() - t0)
        print(f"fold {f} seed {s}: {METRIK} validasi(thr 0.5) = "
              f"{skor(d['label'], d['prob'], METRIK, 0.5):.4f} | {waktu_model[-1]:.0f} s "
              f"({dt:.1f} s/epoch) -> {p_bobot.name}")
        del model
        if device.type == "cuda":
            torch.cuda.empty_cache()
    del Xva
print("model siap:", len(prob_test), "dari", len(daftar_fold) * len(SEEDS))

## OOF: Macro F1 total, per fold, simpangan baku, dan threshold

In [ ]:

oof = (pd.concat(baris_oof, ignore_index=True)
       .groupby(["video", "mulai_frame", "fold", "label"], as_index=False)["prob"].mean())
assert len(oof) == int(val_mask.sum()), f"OOF {len(oof)} baris, diharap {int(val_mask.sum())}"

thr, _ = cari_threshold(oof["label"], oof["prob"], metrik=METRIK)   # threshold HANYA dari OOF train
tabel, ringkasan = skor_per_fold(oof, metrik=METRIK, threshold=thr)
print(f"{METRIK} OOF total        : {skor(oof['label'], oof['prob'], METRIK, thr):.4f}  (thr={thr:.2f})")
print(f"{METRIK} rata-rata fold   : {ringkasan['mean']:.4f}")
print(f"simpangan baku antar fold : {ringkasan['std']:.4f}")
print(ringkas_teks(ringkasan))
print(tabel.to_string(index=False))

p_oof = get_path(cfg, "oof") / "oof_multitask.csv"
p_oof.parent.mkdir(parents=True, exist_ok=True)
oof.to_csv(p_oof, index=False)
p_thr = DIR_MODEL / "threshold_multitask.json"   # terpisah dari threshold.json notebook 05
p_thr.write_text(json.dumps({"model": "07_multitask", "metrik": METRIK, "threshold": float(thr),
                             "skor_oof": float(ringkasan["oof"]), "std_fold": float(ringkasan["std"]),
                             "epoch": int(EPOCH_TETAP), "seeds": list(SEEDS),
                             "n_oof": int(len(oof))}, indent=2), encoding="utf-8")
print("tersimpan:", p_oof, "|", p_thr)

## Test + submission

In [ ]:

# Rata-rata probabilitas 15 model (5 fold x 3 seed), lalu threshold dari OOF train.
# Tiap sequence test diprediksi sendiri dari 30 frame-nya: tidak ada statistik dari data test
# dan tidak ada hubungan antar-sequence (nomor id test tidak dipakai sebagai fitur).
assert len(prob_test) == len(daftar_fold) * len(SEEDS), "belum semua model selesai"
prob_te = np.mean(np.stack([prob_test[k] for k in sorted(prob_test)]), axis=0)
pred = pd.DataFrame({"id": ids_te, "label": (prob_te >= thr).astype(int)})

contoh = pd.read_csv(get_path(cfg, "raw") / "sample_submission.csv")


def kunci(s):
    """Cocokkan id lewat teks, supaya id angka maupun teks sama-sama jalan.
    Kolom id yang ditulis tetap milik sample_submission (urutan dan dtype aslinya)."""
    return s.astype(str).str.strip()


sub = contoh[["id"]].copy()
sub["_k"] = kunci(sub["id"])
sub = sub.merge(pred.assign(_k=kunci(pred["id"]))[["_k", "label"]], on="_k", how="left").drop(columns="_k")

assert sub["label"].notna().all(), (
    "ada id di sample_submission yang tidak punya prediksi | contoh id sample=%r vs emb_test=%r"
    % (contoh["id"].iloc[0], ids_te[0]))
sub["label"] = sub["label"].astype(int)
assert len(sub) == 1515, "jumlah baris %d, seharusnya 1515" % len(sub)
assert sub["id"].tolist() == contoh["id"].tolist(), "urutan id tidak sama dengan sample_submission"
assert set(sub["label"].unique()) <= {0, 1}, "label bukan 0/1"
assert sub["label"].dtype.kind == "i", "label bukan integer"

out = get_path(cfg, "submissions") / "submission_07.csv"
out.parent.mkdir(parents=True, exist_ok=True)
sub.to_csv(out, index=False)
print("Tersimpan:", out, "|", len(sub), "baris | rasio label 1:", round(float(sub["label"].mean()), 4))
print(sub.head(3).to_string(index=False))

## Catat ke logs/experiments.csv

In [ ]:

import datetime

KOLOM_LOG = ["date", "model", "cv_score", "lb_score", "notes"]
baris = {
    "date": datetime.date.today().isoformat(),
    "model": "07_transformer_multitask" + TAG,
    "cv_score": round(float(ringkasan["oof"]), 4),
    "lb_score": "",
    "notes": "epoch=%d thr=%.2f %s per-fold=%.4f+-%.4f n_param=%d loncat=%d seeds=%s "
             "n_train=%d n_oof=%d aux=%.1fperilaku+%.1fsudut" % (
        EPOCH_TETAP, thr, METRIK, ringkasan["mean"], ringkasan["std"], N_PARAM, LONCAT,
        "+".join(str(s) for s in SEEDS), len(idx), len(oof), BOBOT_PERILAKU, BOBOT_SUDUT),
}

p_log = get_path(cfg, "logs")
if ON_KAGGLE:      # folder repo di Kaggle hanya-baca
    p_log = Path("/kaggle/working") / cfg["paths"]["logs"]
log = pd.read_csv(p_log) if p_log.exists() else pd.DataFrame(columns=KOLOM_LOG)
log = pd.concat([log, pd.DataFrame([baris])[KOLOM_LOG]], ignore_index=True)
p_log.parent.mkdir(parents=True, exist_ok=True)
log.to_csv(p_log, index=False)
print("Dicatat di", p_log)
print(log.tail(3).to_string(index=False))

## Waktu jalan

In [ ]:

dt = float(np.mean(detik_epoch)) if detik_epoch else float("nan")
n_epoch_total = len(daftar_fold) * (EPOCH_MAKS + len(SEEDS) * EPOCH_TETAP)
print(f"epoch total (A + B)         : {n_epoch_total}")
if detik_epoch:
    print(f"detik per epoch (tahap A)   : {dt:.1f}")
    print(f"perkiraan waktu total       : {n_epoch_total * dt / 60:.0f} menit "
          f"({n_epoch_total * dt / 3600:.2f} jam)")
else:
    print("detik per epoch (tahap A)   : tidak terukur "
          "(kurva dibaca dari sesi sebelumnya)")
if waktu_model:
    print(f"tahap B terukur             : {len(waktu_model)} model x "
          f"{np.mean(waktu_model):.0f} s = {sum(waktu_model) / 60:.0f} menit")
if detik_epoch and n_epoch_total * dt > 2.5 * 3600:
    print("PERINGATAN: mendekati/melewati 3 jam. Turunkan EPOCH_MAKS atau besarkan LONCAT.")